# Лабораторная работа №3. Демонстрация синтеза речи

Ноутбук фиксирует воспроизводимый сценарий для XTTS-v2: подготовка русского single-speaker датасета, fine-tuning, синтез контрольных фраз и сравнение pretrained/fine-tuned вариантов. В репозиторий добавляется только ноутбук; большие WAV, чекпоинты, TensorBoard-логи и processed-датасеты остаются локальными артефактами.

## 1. Подготовка окружения

Ячейки ниже рассчитаны на запуск из директории `lab3/notebooks`. Команды обучения оставлены как явные shell-команды, чтобы случайно не запустить долгий fine-tuning при открытии ноутбука.

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Audio, Image, Markdown, display

LAB_DIR = Path("..").resolve()
TEXTS_PATH = LAB_DIR / "texts" / "control_texts.json"
XTTS_ORIGINAL_DIR = LAB_DIR / "checkpoints" / "xtts_original"
XTTS_FINETUNE_ROOT = LAB_DIR / "checkpoints" / "xtts_finetune"
XTTS_PRETRAINED_AUDIO = LAB_DIR / "outputs" / "audio" / "xtts_pretrained"
XTTS_FINETUNED_AUDIO = LAB_DIR / "outputs" / "audio" / "xtts_finetuned"
XTTS_COMPARISON_DIR = LAB_DIR / "outputs" / "metrics" / "xtts_comparison"

LAB_DIR

PosixPath('/home/master/alice/OZiR/lab3')

## 2. Подготовка датасета

Используется публичный датасет `niobures/russian-single-speaker-speech-dataset`. Сначала скачиваем подмножество, затем готовим LJSpeech-style split и отдельный Coqui XTTS формат.

In [2]:
# Скачивание исходных фраз и подготовка LJSpeech-style структуры.
# !python ../src/download_dataset.py \
#   --max-samples 1500 \
#   --folder early_short_stories \
#   --output-dir ../data/raw/ru_ss

# !python ../src/prepare_dataset.py \
#   --raw-dir ../data/raw/ru_ss \
#   --output-dir ../data/processed/ru_ss_ljspeech \
#   --sample-rate 22050 \
#   --val-ratio 0.05

# Конвертация в Coqui XTTS формат audio_file|text|speaker_name.
# !python ../src/prepare_xtts_dataset.py \
#   --input-dir ../data/processed/ru_ss_ljspeech \
#   --output-dir ../data/processed/ru_ss_xtts \
#   --speaker-name ru_single_speaker

## 3. Быстрая проверка подготовленных данных

Эта ячейка ничего не генерирует заново: она только показывает, есть ли локальные metadata-файлы и сколько строк попало в train/validation split.

In [3]:
ljspeech_dir = LAB_DIR / "data" / "processed" / "ru_ss_ljspeech"
xtts_dir = LAB_DIR / "data" / "processed" / "ru_ss_xtts"
metadata_files = {
    "LJSpeech train": ljspeech_dir / "metadata_train.csv",
    "LJSpeech val": ljspeech_dir / "metadata_val.csv",
    "XTTS train": xtts_dir / "metadata_train.csv",
    "XTTS eval": xtts_dir / "metadata_eval.csv",
    "speaker reference": xtts_dir / "speaker_ref.txt",
}

for label, path in metadata_files.items():
    if not path.exists():
        print(f"{label}: not found at {path.relative_to(LAB_DIR)}")
        continue
    if path.suffix == ".txt":
        print(f"{label}: {path.read_text(encoding='utf-8').strip()}")
    else:
        rows = pd.read_csv(path, sep="|", header=None if "ljspeech" in str(path).lower() else 0)
        print(f"{label}: {len(rows)} rows")
        display(rows.head(3))

## 4. Fine-tuning XTTS-v2

`--dry-run` проверяет конфиг и структуру датасета без запуска обучения. Полный запуск скачивает исходный XTTS-v2 при необходимости и сохраняет результат в `checkpoints/xtts_finetune/<run-dir>`.

In [4]:
# Проверка без обучения.
# !python ../src/train_xtts.py --config ../configs/xtts_finetune.json --dry-run

# Проверка со скачиванием исходной XTTS-v2 модели.
# !python ../src/train_xtts.py --config ../configs/xtts_finetune.json --dry-run --download-pretrained

# Полный fine-tuning. Запускать только вручную.
# !python ../src/train_xtts.py --config ../configs/xtts_finetune.json

## 5. Синтез контрольных текстов

Для демонстрации сравниваются два режима: исходная `XTTS-v2 pretrained` и дообученная `XTTS-v2 fine-tuned`. Аудиофайлы сохраняются локально в `outputs/audio`, но не добавляются в git.

In [5]:
# Исходная XTTS-v2 без fine-tuning.
# !python ../src/synthesize_xtts.py \
#   --checkpoint ../checkpoints/xtts_original/model.pth \
#   --config ../checkpoints/xtts_original/config.json \
#   --vocab ../checkpoints/xtts_original/vocab.json \
#   --speaker-wav ../data/processed/ru_ss_xtts/speaker_ref.txt \
#   --texts ../texts/control_texts.json \
#   --output-dir ../outputs/audio/xtts_pretrained

# Дообученная XTTS-v2. Замените <run-dir> на фактическую директорию запуска.
# run_dir = "../checkpoints/xtts_finetune/<run-dir>"
# !python ../src/synthesize_xtts.py \
#   --checkpoint "$run_dir/best_model.pth" \
#   --config "$run_dir/config.json" \
#   --vocab "$run_dir/vocab.json" \
#   --speaker-wav ../data/processed/ru_ss_xtts/speaker_ref.txt \
#   --texts ../texts/control_texts.json \
#   --output-dir ../outputs/audio/xtts_finetuned

## 6. Прослушивание локальных примеров

Ячейка ищет одинаковые `*.wav` в папках pretrained и fine-tuned и показывает аудиоплееры. В самом ноутбуке аудио не встраивается, поэтому репозиторий не раздувается WAV-файлами.

In [6]:
def load_control_texts(path: Path) -> dict[str, str]:
    if not path.exists():
        return {}
    data = json.loads(path.read_text(encoding="utf-8"))
    if isinstance(data, dict):
        return {str(key): str(value) for key, value in data.items()}
    return {str(item["id"]): str(item["text"]) for item in data}


def show_audio_pair(item_id: str, text: str | None = None) -> None:
    pretrained = XTTS_PRETRAINED_AUDIO / f"{item_id}.wav"
    finetuned = XTTS_FINETUNED_AUDIO / f"{item_id}.wav"
    display(Markdown(f"### `{item_id}`"))
    if text:
        display(Markdown(text))
    for label, path in [("pretrained", pretrained), ("fine-tuned", finetuned)]:
        if path.exists():
            display(Markdown(f"**{label}**: `{path.relative_to(LAB_DIR)}`"))
            display(Audio(filename=str(path)))
        else:
            print(f"{label}: file not found at {path.relative_to(LAB_DIR)}")

control_texts = load_control_texts(TEXTS_PATH)
for item_id, text in list(control_texts.items())[:5]:
    show_audio_pair(item_id, text)

## 7. Объективное сравнение

Для validation subset синтезируются те же `id`, что есть в `metadata_val.csv`, затем `compare_acoustic_features.py` считает MCD, длительность, RMS, F0, voiced ratio и сохраняет таблицы/графики в `outputs/metrics/xtts_comparison`.

In [ ]:
# Синтез validation subset для объективного сравнения.
# !python ../src/synthesize_xtts.py \
#   --checkpoint ../checkpoints/xtts_original/model.pth \
#   --config ../checkpoints/xtts_original/config.json \
#   --vocab ../checkpoints/xtts_original/vocab.json \
#   --speaker-wav ../data/processed/ru_ss_xtts/speaker_ref.txt \
#   --metadata ../data/processed/ru_ss_ljspeech/metadata_val.csv \
#   --output-dir ../outputs/audio/xtts_pretrained_val

# run_dir = "../checkpoints/xtts_finetune/<run-dir>"
# !python ../src/synthesize_xtts.py \
#   --checkpoint "$run_dir/best_model.pth" \
#   --config "$run_dir/config.json" \
#   --vocab "$run_dir/vocab.json" \
#   --speaker-wav ../data/processed/ru_ss_xtts/speaker_ref.txt \
#   --metadata ../data/processed/ru_ss_ljspeech/metadata_val.csv \
#   --output-dir ../outputs/audio/xtts_finetuned_val

# !python ../src/compare_acoustic_features.py \
#   --metadata ../data/processed/ru_ss_ljspeech/metadata_val.csv \
#   --reference-wavs ../data/processed/ru_ss_ljspeech/wavs \
#   --system pretrained=../outputs/audio/xtts_pretrained_val \
#   --system finetuned=../outputs/audio/xtts_finetuned_val \
#   --output-dir ../outputs/metrics/xtts_comparison

## 8. Просмотр метрик и графиков

Если локально уже рассчитаны `summary.csv/json` и PNG-графики, ячейки ниже покажут их прямо в ноутбуке. Эти файлы остаются локальными и не коммитятся.

In [ ]:
summary_csv = XTTS_COMPARISON_DIR / "summary.csv"
summary_json = XTTS_COMPARISON_DIR / "summary.json"

if summary_csv.exists():
    display(pd.read_csv(summary_csv))
elif summary_json.exists():
    summary = json.loads(summary_json.read_text(encoding="utf-8"))
    display(pd.DataFrame(summary).T)
else:
    print(f"Metrics summary not found in {XTTS_COMPARISON_DIR.relative_to(LAB_DIR)}")

plot_names = [
    "duration_sec.png",
    "rms_mean.png",
    "f0_mean.png",
    "voiced_ratio.png",
    "mcd_to_reference.png",
]
for plot_name in plot_names:
    plot_path = XTTS_COMPARISON_DIR / plot_name
    if plot_path.exists():
        display(Markdown(f"### {plot_name}"))
        display(Image(filename=str(plot_path)))

## 9. Краткие выводы

В коротком лабораторном fine-tuning ожидаемый эффект XTTS-v2 проявляется прежде всего в приближении громкости, длительности и тембра к выбранному диктору. Pretrained baseline уже умеет читать русский текст, но fine-tuned модель должна стабильнее удерживать speaker identity на длинных фразах.

Для отчёта сравниваются:

- субъективное прослушивание пар pretrained/fine-tuned;
- real-time factor из `*_synthesis_metrics.json`;
- validation-признаки из `outputs/metrics/xtts_comparison`;
- TensorBoard loss по GPT fine-tuning.